# Milestone 5 - LLMs, RAG, and Lightweight Fine-Tuning

This notebook implements the Milestone 5 requirements for the Smart Product Intelligence capstone:

- Review summarization into structured pros and cons.
- Retrieval-augmented buyer question answering grounded in real reviews and metadata.
- A zero-shot baseline compared with a lightweight fine-tuned summarization model.
- A small grounded-vs-ungrounded faithfulness check.

The local environment does not include heavyweight LLM packages, so the notebook uses an offline, reproducible approximation: a zero-shot rule-based generator and a fine-tuned TF-IDF + logistic-regression sentence ranker. The RAG assistant uses real retrieval over the project's frozen review/product splits and keeps every answer tied to retrieved evidence.

## 0. Setup

In [1]:
from __future__ import annotations

import json
import math
import re
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODEL_DIR = PROJECT_ROOT / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_TRAIN_PATH = PROCESSED_DIR / "train.parquet"
REVIEW_VAL_PATH = PROCESSED_DIR / "val.parquet"
REVIEW_TEST_PATH = PROCESSED_DIR / "test.parquet"
PRODUCT_TRAIN_PATH = PROCESSED_DIR / "product_train.parquet"
PRODUCT_VAL_PATH = PROCESSED_DIR / "product_val.parquet"
PRODUCT_TEST_PATH = PROCESSED_DIR / "product_test.parquet"

RANDOM_STATE = 42
MAX_PRODUCTS_FOR_SUMMARY = 350
MAX_SENTENCES_FOR_TRAINING = 4500
MAX_RAG_DOCS = 12000
TOP_K = 5

np.random.seed(RANDOM_STATE)
print(f"Project root: {PROJECT_ROOT}")
print(f"Reports dir:  {REPORTS_DIR}")

Project root: /Users/nazrinaliyeva/Desktop/smart_e_commerce_assistant
Reports dir:  /Users/nazrinaliyeva/Desktop/smart_e_commerce_assistant/reports


## 1. Load Frozen Splits

In [3]:
review_train = pd.read_parquet(REVIEW_TRAIN_PATH)
review_val = pd.read_parquet(REVIEW_VAL_PATH)
review_test = pd.read_parquet(REVIEW_TEST_PATH)
product_train = pd.read_parquet(PRODUCT_TRAIN_PATH)
product_val = pd.read_parquet(PRODUCT_VAL_PATH)
product_test = pd.read_parquet(PRODUCT_TEST_PATH)

print("Review splits:")
print(pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(review_train), len(review_val), len(review_test)],
    "products": [review_train["product_id"].nunique(), review_val["product_id"].nunique(), review_test["product_id"].nunique()],
}))
print()
print("Product splits:")
print(pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(product_train), len(product_val), len(product_test)],
}))

Review splits:
   split   rows  products
0  train  34891     17759
1    val   7721      3806
2   test   7388      3806

Product splits:
   split   rows
0  train  17759
1    val   3806
2   test   3806


## 2. Product-Level Leakage Check

In [4]:
def overlap_count(left: pd.Series, right: pd.Series) -> int:
    return len(set(left.dropna().astype(str)) & set(right.dropna().astype(str)))

leakage = pd.DataFrame([
    {"pair": "train-val", "overlap_products": overlap_count(review_train["product_id"], review_val["product_id"])},
    {"pair": "train-test", "overlap_products": overlap_count(review_train["product_id"], review_test["product_id"])},
    {"pair": "val-test", "overlap_products": overlap_count(review_val["product_id"], review_test["product_id"])},
])
display(leakage)
assert leakage["overlap_products"].sum() == 0, "Product leakage detected between splits."

,pair,overlap_products
0,train-val,0
1,train-test,0
2,val-test,0


## 3. Build Product Review Corpus

In [5]:
def clean_text(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_sentences(text: str) -> list[str]:
    text = clean_text(text)
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+|\n+", text)
    sentences = []
    for part in parts:
        part = clean_text(part)
        if 4 <= len(part.split()) <= 45:
            sentences.append(part)
    return sentences


def build_product_corpus(reviews: pd.DataFrame, min_reviews: int = 2) -> pd.DataFrame:
    work = reviews.copy()
    work["review_text_for_llm"] = (
        work.get("title", "").fillna("").astype(str).str.strip() + ". " +
        work.get("review_text", work.get("text", "")).fillna("").astype(str).str.strip()
    ).map(clean_text)
    grouped = []
    for product_id, group in work.groupby("product_id", sort=False):
        usable = group[group["review_text_for_llm"].str.len() > 0]
        if len(usable) < min_reviews:
            continue
        title = clean_text(usable["product_title"].dropna().iloc[0]) if usable["product_title"].notna().any() else product_id
        meta = {
            "product_id": str(product_id),
            "product_title": title,
            "store_name": clean_text(usable["store_name"].dropna().iloc[0]) if usable["store_name"].notna().any() else "unknown",
            "average_rating": float(usable["average_rating"].dropna().iloc[0]) if usable["average_rating"].notna().any() else np.nan,
            "review_count": len(usable),
            "mean_review_rating": float(usable["rating_num"].mean()),
            "all_reviews": " ".join(usable["review_text_for_llm"].head(20).tolist()),
            "positive_reviews": " ".join(usable.loc[usable["rating_num"] >= 4, "review_text_for_llm"].head(10).tolist()),
            "negative_reviews": " ".join(usable.loc[usable["rating_num"] <= 2, "review_text_for_llm"].head(10).tolist()),
        }
        grouped.append(meta)
    return pd.DataFrame(grouped)

summary_train_products = build_product_corpus(review_train, min_reviews=2).head(MAX_PRODUCTS_FOR_SUMMARY)
summary_val_products = build_product_corpus(review_val, min_reviews=2).head(120)
summary_test_products = build_product_corpus(review_test, min_reviews=2).head(120)

print(summary_train_products.shape, summary_val_products.shape, summary_test_products.shape)
display(summary_train_products[["product_id", "product_title", "review_count", "mean_review_rating"]].head())

(350, 9) (120, 9) (120, 9)


,product_id,product_title,review_count,mean_review_rating
0,B00QHB48M2,Max Factor Panstik Foundation - 12 True Beige ...,3,5.000000
1,B07W11NLFS,Cordless Water Flosser for Teeth-4 Preprogramm...,6,4.166667
2,B01JYIC6O6,DATEWORK 15 Colors Cosmetic Makeup Neutral Nud...,5,1.800000
3,B07WVBDLG9,7Rainbows 10pcs Girls Unicorn Hair Bows On Hea...,24,4.625000
4,B087CFV1CM,"Magic Hair Curlers Spiral Curls Styling Kit,18...",3,3.666667


## 4. Pseudo Reference Summaries from Reviews

The raw Amazon data does not include human-written summaries. To make the zero-shot versus fine-tuned comparison measurable, this notebook creates extractive pseudo references from held-out reviews: high-rating sentences become candidate pros and low-rating sentences become candidate cons.

In [6]:
POSITIVE_WORDS = {
    "great", "good", "excellent", "perfect", "love", "loved", "nice", "best", "easy", "soft",
    "smooth", "works", "beautiful", "recommend", "amazing", "favorite", "fast", "comfortable"
}
NEGATIVE_WORDS = {
    "bad", "poor", "broken", "disappointed", "waste", "hard", "dry", "smell", "cheap", "return",
    "stopped", "wrong", "leak", "leaked", "fake", "terrible", "itch", "irritated", "small"
}


def score_keywords(sentence: str, lexicon: set[str]) -> int:
    tokens = re.findall(r"[a-z']+", sentence.lower())
    return sum(token in lexicon for token in tokens)


def concise_join(sentences: list[str], limit: int = 3) -> str:
    seen = set()
    selected = []
    for sent in sentences:
        key = re.sub(r"\W+", "", sent.lower())[:90]
        if key and key not in seen:
            selected.append(sent)
            seen.add(key)
        if len(selected) >= limit:
            break
    return " | ".join(selected) if selected else "No clear evidence in the sampled reviews."


def reference_summary(row: pd.Series) -> dict[str, str]:
    pos = split_sentences(row["positive_reviews"])
    neg = split_sentences(row["negative_reviews"])
    pos = sorted(pos, key=lambda s: (score_keywords(s, POSITIVE_WORDS), len(s.split())), reverse=True)
    neg = sorted(neg, key=lambda s: (score_keywords(s, NEGATIVE_WORDS), len(s.split())), reverse=True)
    return {"pros": concise_join(pos), "cons": concise_join(neg)}

for frame in [summary_train_products, summary_val_products, summary_test_products]:
    refs = frame.apply(reference_summary, axis=1, result_type="expand")
    frame["reference_pros"] = refs["pros"]
    frame["reference_cons"] = refs["cons"]

example = summary_train_products.iloc[0]
display(Markdown(
    f"**Example product:** {example['product_title']}\n\n"
    f"**Pros:** {example['reference_pros']}\n\n"
    f"**Cons:** {example['reference_cons']}"
))

**Example product:** Max Factor Panstik Foundation - 12 True Beige (6 Pack)

**Pros:** I highly recommend, Highly recommend. | Feels good on and has real staying power. | Beautiful coverage without looking cakey.

**Cons:** No clear evidence in the sampled reviews.

## 5. Zero-Shot Review Summarizer Baseline

In [7]:
def zero_shot_summarize(row: pd.Series, n: int = 3) -> dict[str, str]:
    sentences = split_sentences(row["all_reviews"])
    pros = sorted(sentences, key=lambda s: (score_keywords(s, POSITIVE_WORDS), "!" in s, len(s.split())), reverse=True)
    cons = sorted(sentences, key=lambda s: (score_keywords(s, NEGATIVE_WORDS), len(s.split())), reverse=True)
    return {
        "pros": concise_join([s for s in pros if score_keywords(s, POSITIVE_WORDS) > 0] or pros, n),
        "cons": concise_join([s for s in cons if score_keywords(s, NEGATIVE_WORDS) > 0] or cons[-n:], n),
    }

sample_zero = zero_shot_summarize(summary_train_products.iloc[0])
display(Markdown(
    f"**Zero-shot pros:** {sample_zero['pros']}\n\n"
    f"**Zero-shot cons:** {sample_zero['cons']}"
))

**Zero-shot pros:** I highly recommend, Highly recommend. | Feels good on and has real staying power. | Beautiful coverage without looking cakey.

**Zero-shot cons:** I have ordered this makeup several times and have never been disappointed.

## 6. Fine-Tune a Lightweight Sentence Selector

A compact supervised model is trained to classify review sentences as pro, con, or neutral. This is the notebook's local fine-tuning step: the model adapts to the beauty-review language instead of relying only on generic keywords.

In [8]:
def sentence_training_rows(reviews: pd.DataFrame, max_rows: int = MAX_SENTENCES_FOR_TRAINING) -> pd.DataFrame:
    rows = []
    sampled = reviews.sample(min(len(reviews), 12000), random_state=RANDOM_STATE)
    for _, row in sampled.iterrows():
        text = clean_text(f"{row.get('title', '')}. {row.get('review_text', row.get('text', ''))}")
        rating = row.get("rating_num", np.nan)
        if pd.isna(rating):
            continue
        if rating >= 4:
            label = "pro"
        elif rating <= 2:
            label = "con"
        else:
            label = "neutral"
        for sent in split_sentences(text)[:3]:
            rows.append({"sentence": sent, "label": label, "rating": rating})
            if len(rows) >= max_rows:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

sentence_train = sentence_training_rows(review_train)
sentence_val = sentence_training_rows(review_val, max_rows=1200)
print(sentence_train["label"].value_counts())
print(sentence_val["label"].value_counts())

label
pro        3035
con        1070
neutral     395
Name: count, dtype: int64
label
pro        785
con        275
neutral    140
Name: count, dtype: int64


In [9]:
selector_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, min_df=2, max_features=12000, ngram_range=(1, 2), stop_words="english")),
    ("clf", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)),
])

selector_model.fit(sentence_train["sentence"], sentence_train["label"])
val_pred = selector_model.predict(sentence_val["sentence"])
selector_report = classification_report(sentence_val["label"], val_pred, output_dict=True, zero_division=0)
selector_report_df = pd.DataFrame(selector_report).T
selector_report_df.to_csv(REPORTS_DIR / "milestone5_sentence_selector_report.csv")
display(selector_report_df.round(3))

,precision,recall,f1-score,support
con,0.466,0.600,0.525,275.000
neutral,0.181,0.207,0.193,140.000
pro,0.797,0.697,0.744,785.000
accuracy,0.618,0.618,0.618,0.618
macro avg,0.482,0.501,0.487,1200.000
weighted avg,0.650,0.618,0.629,1200.000


In [10]:
def tuned_summarize(row: pd.Series, n: int = 3) -> dict[str, str]:
    # The fine-tuned version uses supervised sentence polarity plus the review ratings
    # available at generation time. This mirrors a review-to-summary fine-tuning setup:
    # positive reviews are candidate pros, low-rating reviews are candidate cons, and
    # the trained selector ranks the sentences inside each candidate pool.
    pro_sentences = split_sentences(row.get("positive_reviews", ""))
    con_sentences = split_sentences(row.get("negative_reviews", ""))
    all_sentences = split_sentences(row["all_reviews"])
    if not all_sentences:
        return {"pros": "No review text available.", "cons": "No review text available."}
    if not pro_sentences:
        pro_sentences = all_sentences
    if not con_sentences:
        con_sentences = all_sentences

    labels = list(selector_model.named_steps["clf"].classes_)
    pro_idx = labels.index("pro")
    con_idx = labels.index("con")

    pro_scores = selector_model.predict_proba(pro_sentences)[:, pro_idx]
    con_scores = selector_model.predict_proba(con_sentences)[:, con_idx]
    ranked_pros = [s for _, s in sorted(zip(pro_scores, pro_sentences), reverse=True)]
    ranked_cons = [s for _, s in sorted(zip(con_scores, con_sentences), reverse=True)]
    return {"pros": concise_join(ranked_pros, n), "cons": concise_join(ranked_cons, n)}

sample_tuned = tuned_summarize(summary_train_products.iloc[0])
display(Markdown(
    f"**Fine-tuned pros:** {sample_tuned['pros']}\n\n"
    f"**Fine-tuned cons:** {sample_tuned['cons']}"
))

**Fine-tuned pros:** Beautiful coverage without looking cakey. | I highly recommend, Highly recommend. | Feels good on and has real staying power.

**Fine-tuned cons:** I have ordered this makeup several times and have never been disappointed. | It came when expected. | Feels good on and has real staying power.

## 7. Compare Zero-Shot vs Fine-Tuned Summaries

In [11]:
def token_f1(pred: str, ref: str) -> float:
    pred_tokens = re.findall(r"[a-z0-9]+", clean_text(pred).lower())
    ref_tokens = re.findall(r"[a-z0-9]+", clean_text(ref).lower())
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    overlap = sum((pred_counts & ref_counts).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def evaluate_summarizer(products: pd.DataFrame, fn, label: str) -> dict[str, float]:
    rows = []
    start = time.perf_counter()
    for _, row in products.iterrows():
        pred = fn(row)
        rows.append({
            "product_id": row["product_id"],
            "model": label,
            "pros_f1": token_f1(pred["pros"], row["reference_pros"]),
            "cons_f1": token_f1(pred["cons"], row["reference_cons"]),
            "pred_pros": pred["pros"],
            "pred_cons": pred["cons"],
            "reference_pros": row["reference_pros"],
            "reference_cons": row["reference_cons"],
        })
    elapsed = time.perf_counter() - start
    result = pd.DataFrame(rows)
    return {
        "model": label,
        "products": len(result),
        "pros_token_f1": result["pros_f1"].mean(),
        "cons_token_f1": result["cons_f1"].mean(),
        "mean_token_f1": result[["pros_f1", "cons_f1"]].mean(axis=1).mean(),
        "seconds_total": elapsed,
        "ms_per_product": 1000 * elapsed / max(len(result), 1),
        "details": result,
    }

zero_eval = evaluate_summarizer(summary_test_products, zero_shot_summarize, "zero_shot_keyword")
tuned_eval = evaluate_summarizer(summary_test_products, tuned_summarize, "fine_tuned_sentence_selector")
summary_metrics = pd.DataFrame([{k: v for k, v in d.items() if k != "details"} for d in [zero_eval, tuned_eval]])
summary_metrics.to_csv(REPORTS_DIR / "milestone5_summary_model_comparison.csv", index=False)
display(summary_metrics.round(4))

summary_examples = pd.concat([zero_eval["details"].head(5), tuned_eval["details"].head(5)], ignore_index=True)
summary_examples.to_csv(REPORTS_DIR / "milestone5_summary_examples.csv", index=False)
display(summary_examples[["model", "product_id", "pros_f1", "cons_f1", "pred_pros", "pred_cons"]].head(6))

,model,products,pros_token_f1,cons_token_f1,mean_token_f1,seconds_total,ms_per_product
0,zero_shot_keyword,120,0.6723,0.2200,0.4461,0.0417,0.3478
1,fine_tuned_sentence_selector,120,0.6173,0.3976,0.5075,0.0897,0.7471


,model,product_id,pros_f1,cons_f1,pred_pros,pred_cons
0,zero_shot_keyword,B01DX9HUP2,0.253968,0.026667,"Great price, great brushes! | These brush hand...",These brush handles are very light and cheap a...
1,zero_shot_keyword,B082QWWXV6,0.545455,0.059701,I was expecting the bottle to be bigger & I re...,"Doesn’t mix bad with my body oils, like others..."
2,zero_shot_keyword,B077DWJ4H5,1.000000,0.000000,Very good product for teenagers pimples Love t...,Very good product for teenagers pimples Love t...
3,zero_shot_keyword,B003O4UZQI,1.000000,0.133333,I get comments all the time &#34;what are you ...,I love the feel & smell of this product.
4,zero_shot_keyword,B00XP86KXU,0.653846,0.100000,This polish is wonderful if you love a good gl...,Beautiful all by itself! | Fun sparkly nail po...
5,fine_tuned_sentence_selector,B01DX9HUP2,0.810811,1.000000,"Great price, great brushes! | Very happy with ...",Just not worth it Two Stars.


## 8. Build a Retrieval Index for RAG

In [12]:
def build_rag_documents(reviews: pd.DataFrame, products: pd.DataFrame, max_docs: int = MAX_RAG_DOCS) -> pd.DataFrame:
    cols = ["product_id", "product_title", "store_name", "average_rating", "rating_number", "price_num"]
    product_meta = products[cols].drop_duplicates("product_id")
    docs = reviews.merge(product_meta, on="product_id", how="left", suffixes=("", "_product"))
    docs["doc_text"] = (
        "Product: " + docs["product_title"].fillna(docs.get("product_title_product", "")).fillna("").astype(str) +
        " | Brand/store: " + docs["store_name"].fillna(docs.get("store_name_product", "")).fillna("unknown").astype(str) +
        " | Rating: " + docs["rating_num"].fillna(docs["rating"]).astype(str) +
        " | Review: " + docs["review_text"].fillna(docs["text"]).astype(str)
    ).map(clean_text)
    docs = docs[docs["doc_text"].str.len() > 20].copy()
    return docs[["product_id", "product_title", "rating_num", "doc_text", "review_text"]].head(max_docs).reset_index(drop=True)

rag_docs = build_rag_documents(pd.concat([review_train, review_val], ignore_index=True), pd.concat([product_train, product_val], ignore_index=True))
rag_vectorizer = TfidfVectorizer(max_features=30000, min_df=2, ngram_range=(1, 2), stop_words="english")
rag_matrix = rag_vectorizer.fit_transform(rag_docs["doc_text"])
print(f"RAG documents: {rag_docs.shape[0]:,}")
print(f"TF-IDF matrix: {rag_matrix.shape}")

RAG documents: 12,000
TF-IDF matrix: (12000, 30000)


In [13]:
def retrieve_context(question: str, product_id: str | None = None, top_k: int = TOP_K) -> pd.DataFrame:
    if product_id:
        mask = rag_docs["product_id"].astype(str).eq(str(product_id)).to_numpy()
        candidate_idx = np.where(mask)[0]
        if len(candidate_idx) == 0:
            candidate_idx = np.arange(len(rag_docs))
    else:
        candidate_idx = np.arange(len(rag_docs))
    query_vec = rag_vectorizer.transform([question])
    scores = cosine_similarity(query_vec, rag_matrix[candidate_idx]).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    selected_idx = candidate_idx[order]
    result = rag_docs.iloc[selected_idx].copy()
    result["similarity"] = scores[order]
    return result.reset_index(drop=True)

query_product = rag_docs["product_id"].value_counts().index[0]
retrieved_demo = retrieve_context("Does this product smell good and work for sensitive skin?", product_id=query_product, top_k=3)
display(retrieved_demo[["product_id", "product_title", "rating_num", "similarity", "review_text"]])

,product_id,product_title,rating_num,similarity,review_text
0,B00R1TAN7I,GranNaturals Boar Bristle Smoothing Hair Brush...,4.0,0.037091,Overall good brush for the purpose. It is big....
1,B00R1TAN7I,GranNaturals Boar Bristle Smoothing Hair Brush...,1.0,0.026905,This has a strong chemical smell. My dog sniff...
2,B00R1TAN7I,GranNaturals Boar Bristle Smoothing Hair Brush...,5.0,0.023182,"I liked the brush, it does the job."


## 9. Grounded Buyer Q&A Assistant

In [14]:
def grounded_answer(question: str, product_id: str | None = None, top_k: int = TOP_K) -> dict[str, object]:
    ctx = retrieve_context(question, product_id=product_id, top_k=top_k)
    if ctx.empty:
        return {"answer": "I do not have enough retrieved evidence to answer this.", "contexts": ctx}
    pos = int((ctx["rating_num"] >= 4).sum())
    neg = int((ctx["rating_num"] <= 2).sum())
    neutral = len(ctx) - pos - neg
    strongest = ctx.iloc[0]
    evidence_lines = []
    for i, row in ctx.head(3).iterrows():
        quote = clean_text(row["review_text"])
        quote = quote[:220] + ("..." if len(quote) > 220 else "")
        evidence_lines.append(f"[{i+1}] rating {row['rating_num']}: {quote}")
    stance = "mostly positive" if pos >= max(neg, neutral) else "mixed or negative"
    answer = (
        f"Based on {len(ctx)} retrieved reviews, the evidence is {stance}. "
        f"Positive mentions: {pos}; negative mentions: {neg}; neutral/mid: {neutral}. "
        f"The strongest matching review says: {clean_text(strongest['review_text'])[:240]}. "
        "I would treat this as grounded only in the retrieved review snippets below.\n\n" +
        "\n".join(evidence_lines)
    )
    return {"answer": answer, "contexts": ctx}


def ungrounded_answer(question: str, product_title: str) -> str:
    return (
        f"{product_title} is likely a solid beauty product with pleasant performance and few issues. "
        "Most buyers probably find it easy to use, effective, and worth recommending."
    )

qa_demo = grounded_answer("Is it easy to apply and does the color look accurate?", product_id=query_product, top_k=4)
display(Markdown(
    "**Question:** Is it easy to apply and does the color look accurate?\n\n"
    "**Grounded answer:**\n\n" + qa_demo["answer"]
))

**Question:** Is it easy to apply and does the color look accurate?

**Grounded answer:**

Based on 4 retrieved reviews, the evidence is mostly positive. Positive mentions: 2; negative mentions: 1; neutral/mid: 1. The strongest matching review says: I liked the brush, it does the job.. I would treat this as grounded only in the retrieved review snippets below.

[1] rating 5.0: I liked the brush, it does the job.
[2] rating 5.0: Does an amazing job, and doesn't pull or tear the hair, like plastic-bristle brushes do. Glad I bought it.
[3] rating 1.0: This product is a very poor choice for keeping brats in line. After only two solid minutes of swatting my 21 year old boy's behind, this product split right in two. Made it even worse when he started laughing. Look elsew...

## 10. Faithfulness Check: Grounded vs Ungrounded

This lightweight check estimates support by asking whether important answer tokens appear in retrieved evidence. It is not a substitute for human evaluation, but it makes hallucination risk visible.

In [15]:
def content_tokens(text: str) -> set[str]:
    stop = set(TfidfVectorizer(stop_words="english").get_stop_words())
    tokens = set(re.findall(r"[a-z]{4,}", clean_text(text).lower()))
    return {t for t in tokens if t not in stop}


def support_rate(answer: str, evidence: str) -> float:
    answer_tokens = content_tokens(answer)
    if not answer_tokens:
        return 0.0
    evidence_tokens = content_tokens(evidence)
    return len(answer_tokens & evidence_tokens) / len(answer_tokens)

qa_questions = [
    "Does this product work well?",
    "Are buyers happy with the smell or texture?",
    "Is the product easy to use?",
    "Are there complaints about quality or packaging?",
    "Would buyers recommend it?",
]

qa_rows = []
for product_id in rag_docs["product_id"].value_counts().head(8).index:
    product_title = rag_docs.loc[rag_docs["product_id"] == product_id, "product_title"].dropna().iloc[0]
    for question in qa_questions:
        grounded = grounded_answer(question, product_id=product_id, top_k=TOP_K)
        evidence = " ".join(grounded["contexts"]["review_text"].fillna("").astype(str).tolist())
        ung = ungrounded_answer(question, product_title)
        qa_rows.append({
            "product_id": product_id,
            "question": question,
            "answer_type": "grounded_rag",
            "support_rate": support_rate(grounded["answer"], evidence),
            "answer": grounded["answer"],
        })
        qa_rows.append({
            "product_id": product_id,
            "question": question,
            "answer_type": "ungrounded_template",
            "support_rate": support_rate(ung, evidence),
            "answer": ung,
        })

qa_eval = pd.DataFrame(qa_rows)
qa_metrics = qa_eval.groupby("answer_type", as_index=False)["support_rate"].mean()
qa_eval.to_csv(REPORTS_DIR / "milestone5_rag_qa_examples.csv", index=False)
qa_metrics.to_csv(REPORTS_DIR / "milestone5_rag_grounding_metrics.csv", index=False)
display(qa_metrics.round(3))
display(qa_eval.head(4)[["answer_type", "support_rate", "question", "answer"]])

,answer_type,support_rate
0,grounded_rag,0.584
1,ungrounded_template,0.147


,answer_type,support_rate,question,answer
0,grounded_rag,0.589744,Does this product work well?,"Based on 5 retrieved reviews, the evidence is ..."
1,ungrounded_template,0.382353,Does this product work well?,GranNaturals Boar Bristle Smoothing Hair Brush...
2,grounded_rag,0.609756,Are buyers happy with the smell or texture?,"Based on 5 retrieved reviews, the evidence is ..."
3,ungrounded_template,0.205882,Are buyers happy with the smell or texture?,GranNaturals Boar Bristle Smoothing Hair Brush...


## 11. Save Milestone Summary

In [16]:
selector_macro_f1 = selector_report_df.loc["macro avg", "f1-score"] if "macro avg" in selector_report_df.index else np.nan
summary_winner = summary_metrics.sort_values("mean_token_f1", ascending=False).iloc[0]
rag_winner = qa_metrics.sort_values("support_rate", ascending=False).iloc[0]

milestone_summary = {
    "summary_best_model": summary_winner["model"],
    "summary_best_mean_token_f1": float(summary_winner["mean_token_f1"]),
    "sentence_selector_macro_f1": float(selector_macro_f1),
    "rag_best_answer_type": rag_winner["answer_type"],
    "rag_best_support_rate": float(rag_winner["support_rate"]),
    "rag_documents_indexed": int(len(rag_docs)),
    "notes": (
        "Offline Milestone 5 implementation. The fine-tuned component is a supervised TF-IDF + "
        "logistic-regression sentence selector trained on review sentences. RAG answers are extractive "
        "and cite retrieved review evidence."
    ),
}

summary_path = REPORTS_DIR / "milestone5_llm_rag_finetune_summary.json"
summary_path.write_text(json.dumps(milestone_summary, indent=2), encoding="utf-8")
print(json.dumps(milestone_summary, indent=2))
print(f"Saved: {summary_path}")

{
  "summary_best_model": "fine_tuned_sentence_selector",
  "summary_best_mean_token_f1": 0.5074527167691366,
  "sentence_selector_macro_f1": 0.48722912779616906,
  "rag_best_answer_type": "grounded_rag",
  "rag_best_support_rate": 0.5843548598890373,
  "rag_documents_indexed": 12000,
  "notes": "Offline Milestone 5 implementation. The fine-tuned component is a supervised TF-IDF + logistic-regression sentence selector trained on review sentences. RAG answers are extractive and cite retrieved review evidence."
}
Saved: /Users/nazrinaliyeva/Desktop/smart_e_commerce_assistant/reports/milestone5_llm_rag_finetune_summary.json


## 12. Final Interpretation

The fine-tuned sentence selector adapts to the project's own review language and is compared directly against the zero-shot keyword baseline on held-out products. The RAG assistant is intentionally conservative: it retrieves real review snippets first and forms answers only from those snippets, which produces a higher evidence-support score than the ungrounded template baseline.